In [1]:
import torch
import timm
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
%load_ext autoreload
 
%autoreload 2

model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True)
# model.load_state_dict(torch.load("swin/Swin-Transformer/checkpoints/swin_tiny_patch4_window7_224.pth"))
model.eval()


# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# model.eval()
# Validation function for PyTorch model with progress bar
def validate_pytorch(model, dataloader, device):
    model.eval()
    top1_correct = 0
    top5_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating PyTorch Model", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, pred_top1 = outputs.topk(1, dim=1)
            _, pred_top5 = outputs.topk(5, dim=1)

            top1_correct += (pred_top1.squeeze() == labels).sum().item()
            top5_correct += sum([labels[i] in pred_top5[i] for i in range(len(labels))])
            total += labels.size(0)

    top1_acc = 100 * top1_correct / total
    top5_acc = 100 * top5_correct / total
    return top1_acc, top5_acc

# Define ImageNet validation dataset and DataLoader
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder("/media/bmw/datasets/imagenet-1k/val", transform=transform)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# Validate model
# top1_acc, top5_acc = validate_pytorch(model, dataloader, device)
# print(f"Top-1 Accuracy: {top1_acc:.2f}%")
# print(f"Top-5 Accuracy: {top5_acc:.2f}%")


/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/onnxscript/converter.py:823: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [2]:
from aimet_torch.model_preparer import prepare_model
from aimet_torch.model_validator.model_validator import ModelValidator
 
print("\nPreparing and validating the model for quantization...")
prepared_model = prepare_model(model)

invalid_layers = ModelValidator.validate_model(prepared_model, model_input=torch.randn(1, 3, 224, 224).to(device))

if not (invalid_layers):
    print("\n❌ Model contains unsupported layers for AIMET quantization:")
else:
    print("\n✅ Model is valid for AIMET quantization!")



2025-02-28 14:33:16,452 - root - INFO - AIMET


/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:110: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  torch.has_cuda,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:111: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  torch.has_cudnn,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:117: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  torch.has_mps,
/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/overrides.py:118: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  torch.has_mkldnn,



Preparing and validating the model for quantization...
2025-02-28 14:33:18,396 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.module_floordiv} 
2025-02-28 14:33:18,396 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.module_floordiv_1} 
2025-02-28 14:33:18,397 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_floordiv_2} 
2025-02-28 14:33:18,397 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_mul} 
2025-02-28 14:33:18,398 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_matmul} 
2025-02-28 14:33:18,398 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.module_add} 
2025-02-28 14:33:18,398 - ModelPreparer - INFO - Functional         : Adding new module for node: {layers.0.blocks.0.attn.

In [3]:
print(type(prepared_model))

<class 'torch.fx.graph_module.GraphModule.__new__.<locals>.GraphModuleImpl'>


In [4]:

from aimet_torch.meta.connectedgraph import ConnectedGraph
 
operations = [
            "floor_divide", "remainder", "pad", "dropout", "Add", "linear", "pow",
            "Mul", "matmul", "softmax", "rsub", "roll", "fill", "masked_fill",
            "new_zeros", "sub", "Concat"
        ]

ConnectedGraph.math_invariant_types.update(operations)

In [5]:
invalid_layers = ModelValidator.validate_model(prepared_model, model_input=torch.randn(1, 3, 224, 224).to(device))

if not (invalid_layers):
    print("\n❌ Model contains unsupported layers for AIMET quantization:")
else:
    print("\n✅ Model is valid for AIMET quantization!")


2025-02-28 14:33:36,509 - Utils - INFO - Running validator check <function validate_for_reused_modules at 0x7fbf502617e0>
2025-02-28 14:33:36,534 - Utils - INFO - Running validator check <function validate_for_missing_modules at 0x7fbf50261ab0>
2025-02-28 14:33:50,612 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-02-28 14:33:50,645 - Utils - INFO - All validation checks passed.

✅ Model is valid for AIMET quantization!


In [6]:

# @QuantizationMixin.implements(DropPath)
# class QuantizedDropPath(QuantizationMixin, DropPath):
#     def __quant_init__(self):
#         super().__quant_init__()

#         # Declare the number of input/output quantizers
#         self.input_quantizers = torch.nn.ModuleList([None])
#         self.output_quantizers = torch.nn.ModuleList([None])

#     def forward(self, x):
#         # Quantize input tensors
#         if self.input_quantizers[0]:
#             x = self.input_quantizers[0](x)

#         # Run forward with quantized inputs and parameters
#         with self._patch_quantized_parameters():
#             ret = super().forward(x)

#         # Quantize output tensors
#         if self.output_quantizers[0]:
#             ret = self.output_quantizers[0](ret)

#         return ret
    


# @QuantizationMixin.implements(FastAdaptiveAvgPool)
# class QuantizedFastAdaptiveAvgPool(QuantizationMixin, FastAdaptiveAvgPool):
#     def __quant_init__(self):
#         super().__quant_init__()

#         # Declare the number of input/output quantizers
#         self.input_quantizers = torch.nn.ModuleList([None])
#         self.output_quantizers = torch.nn.ModuleList([None])

#     def forward(self, x):
#         # Quantize input tensors
#         if self.input_quantizers[0]:
#             x = self.input_quantizers[0](x)

#         # Run forward with quantized inputs and parameters
#         with self._patch_quantized_parameters():
#             ret = super().forward(x)

#         # Quantize output tensors
#         if self.output_quantizers[0]:
#             ret = self.output_quantizers[0](ret)

#         return ret


In [7]:
dummy_input = torch.rand(1, 3, 224, 224).to(device)

In [8]:
# from aimet_torch.batch_norm_fold import fold_all_batch_norms

# import aimet_torch.quantsim as qsim
# def apply_aimet_quantization(model, device, dataloader):
#     dummy_input = torch.rand(1, 3, 224, 224).to(device)
    
#     # Print model before BN folding
#     print("\nModel before BatchNorm folding:")
#     print(model)
    
#     fold_all_batch_norms(model, input_shapes=(1, 3, 224, 224), dummy_input=dummy_input)
    
#     # Print model after BN folding
#     print("\nModel after BatchNorm folding:")
#     print(model)
    
#     quant_sim = qsim.QuantizationSimModel( 
#         model, 
#         dummy_input=dummy_input, 
#         quant_scheme= 'tf_enhanced',
#         rounding_mode= 'nearest',
#         default_param_bw= 8,
#         default_output_bw = 8
#     )
    
#     # Use real calibration data for encoding computation
#     def forward_pass(model, dataloader):
#         model.eval()
#         with torch.no_grad():
#             for i, (images, _) in enumerate(dataloader):
#                 print(i , images[0].shape)
#                 if i >= 10:  # Use only first 10 batches for calibration
                    
#                     break
#                 model(images.to(device))
#                 # print()
    
#     quant_sim.compute_encodings(forward_pass, dataloader)


#     return quant_sim.model


In [9]:
# quantized_model = apply_aimet_quantization(prepared_model,  device, dataloader)

# # Validate model
# top1_acc, top5_acc = validate_pytorch(quantized_model, dataloader, device)
# print(f"Top-1 Accuracy: {top1_acc:.2f}%")
# print(f"Top-5 Accuracy: {top5_acc:.2f}%")

In [10]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset
from torch.utils.data import Dataset

In [11]:
TRAIN_DIR="/media/bmw/datasets/imagenet-1k/train"
VAL_DIR="/media/bmw/datasets/imagenet-1k/val"

def load_labbeled_data(num_of_classes:int =50, images_per_class:int=10,batch_size:int =128,isTrain:bool =False) -> DataLoader :
    
    tranform_data= transforms.Compose([
                        transforms.Resize((224,224)),
                        transforms.ToTensor()
    ])


    if isTrain:
        DATA_DIR=TRAIN_DIR
    else:
        DATA_DIR=VAL_DIR
        
    dataset = ImageFolder(
                    root=DATA_DIR,
                    transform=tranform_data
    )


    all_classes=dataset.classes

    # Filter dataset to include only a subset of classes
    selected_classes = all_classes[:num_of_classes]

    # Create a mapping of selected class names to their indices
    selected_class_indices = [dataset.class_to_idx[class_name] for class_name in selected_classes]

    # Get indices of samples belonging to selected classes
    selected_indices = []
    class_counts = {idx: 0 for idx in selected_class_indices}

    for idx, (_, label) in enumerate(dataset.samples):
        if label in selected_class_indices and class_counts[label] < images_per_class:
            selected_indices.append(idx)
            class_counts[label] += 1

    # Create a subset dataset
    subset_dataset = Subset(dataset, selected_indices)

    dataloader = DataLoader (
    subset_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory= True
    )
    
    print(f"Dataset size {len(subset_dataset)} DataLoader {len(dataloader)}")
    
    return subset_dataset,dataloader

    

def load_unlabelled_data(num_of_classes:int =50, images_per_class:int=10,batch_size:int =128) -> DataLoader :
    
    
    dataset,dataloader =load_labbeled_data(num_of_classes,images_per_class,batch_size)

    class UnlabelledDataset(ImageFolder):
        def __init__(self, dataset):
            self._dataset = dataset

        def __len__(self):
            return len(self._dataset)

        def __getitem__(self, index):
            images, _ = self._dataset[index]
            return images
        
    unlabelled_dataset=UnlabelledDataset(dataset)
    
    unlabelled_dataloader= DataLoader (
    unlabelled_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=8,
    pin_memory= True
    )
    
    return unlabelled_dataset,unlabelled_dataloader





In [18]:
from typing import Optional
from aimet_torch.utils import in_eval_mode, get_device

from tqdm import tqdm

def eval_callback(model: torch.nn.Module,num_samples: Optional[int] = None) -> float:
    dataset,dataloader=load_labbeled_data(10,10,32)
    device = get_device(model)
    
    correct = 0
    with in_eval_mode(model), torch.no_grad():
        for image, label in tqdm(dataloader):
            image = image.to(device)
            label = label.to(device)
            logits = model(image)
            top1 = logits.topk(k=1).indices
            correct += (top1 == label.view_as(top1)).sum()

    return int(correct) / len(dataset)

In [19]:
eval_callback(prepared_model)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  4.76it/s]


0.86

In [20]:

dataset,dataloader =load_unlabelled_data(20,20,32)

Dataset size 400 DataLoader 13


In [21]:
# import aimet_torch.auto_quant as auto_quant
# autoquant_model = auto_quant.AutoQuantWithAutoMixedPrecision(
    
#     model= prepared_model,
#     data_loader=dataloader,
#     eval_callback= lambda model: validate_pytorch(model, imagenet_val_loader,),
#     param_bw=8,
#     output_bw=8,
#     dummy_input= torch.randn((1, 3, 224, 224)).to(device),
#     results_dir="/media/bmw/harisharajan_r_r/swin/artifacts"
# )
from aimet_torch.auto_quant import AutoQuantWithAutoMixedPrecision

auto_quant= AutoQuantWithAutoMixedPrecision(prepared_model,dummy_input=torch.randn(1, 3, 224, 224).to(device),data_loader=dataloader,eval_callback=eval_callback)
print(auto_quant)

In [22]:
sim, initial_accuracy =  auto_quant.run_inference()
print(f"- Quantized Accuracy (before optimization): {initial_accuracy}")

2025-02-28 14:38:26,769 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_mul} 


2025-02-28 14:38:26,772 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_matmul} 


2025-02-28 14:38:26,777 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_add} 


2025-02-28 14:38:26,780 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_matmul_1} 


2025-02-28 14:38:26,782 - ModelPreparer - INFO - Functional         : Adding new module for node: {module_add_1} 


...

2025-02-28 14:41:58,156 - AutoQuant - INFO - Session finished: Batchnorm Folding. (eval score: 0.850000)


/ Batchnorm Folding


2025-02-28 14:42:19,897 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-02-28 14:42:20,215 - Quant - INFO - No config file provided, defaulting to config file at /home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/aimet_common/quantsim_config/default_config.json
2025-02-28 14:42:20,255 - Quant - INFO - Unsupported op type Squeeze
2025-02-28 14:42:20,256 - Quant - INFO - Unsupported op type Mean
2025-02-28 14:42:20,289 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:default
2025-02-28 14:42:20,293 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-02-28 14:42:20,294 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-02-28 14:42:20,294 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
202

100%|██████████| 13/13 [01:43<00:00,  7.94s/it]


- Quantized Accuracy (before optimization): 0.85


Applying AdaRound 

In [23]:
from aimet_torch.adaround.adaround_weight import AdaroundParameters


adaround_params = AdaroundParameters(dataloader, num_batches=len(dataloader), default_num_iterations=2000)
auto_quant.set_adaround_params(adaround_params)

In [24]:
model, optimized_accuracy, encoding_path , pareto_front= auto_quant.optimize(allowed_accuracy_drop=0.01)
print(f"- Quantized Accuracy (after optimization):  {optimized_accuracy}")

2025-02-28 14:45:28,029 - AutoQuant - INFO - Starting AutoQuant


Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:02<00:00,  1.92it/s]

2025-02-28 14:45:30,885 - AutoQuant - INFO - Target eval score: 0.850000
2025-02-28 14:45:30,885 - AutoQuant - INFO - FP32 eval score (W32A32): 0.860000



- Prepare Model


...

2025-02-28 14:53:02,795 - AutoQuant - INFO - Evaluation finished: W32A8 (eval score: 0.870000)


/ W32 Evaluation


2025-02-28 14:53:03,192 - AutoQuant - INFO - Session finished: Batchnorm Folding. (eval score: 0.850000)


- Batchnorm Folding


2025-02-28 14:53:03,690 - AutoQuant - INFO - Best eval score: 0.850000
- Quantized Accuracy (after optimization):  0.85


In [25]:
device

device(type='cuda')

In [26]:
sim.model.to(device)

GraphModule(
  (patch_embed): Module(
    (proj): QuantizedConv2d(
      3, 96, kernel_size=(4, 4), stride=(4, 4)
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
    (norm): QuantizedLayerNorm(
      (96,), eps=1e-05, elementwise_affine=True
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): None
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
  )
  (pos_drop): QuantizedDropout(
    p=0.0, inplace=Fa

In [27]:
print(next(sim.model.parameters()).device)


cuda:0


In [28]:
sim.export(
    "/media/bmw/harisharajan_r_r/swin/artifacts/autoquant", filename_prefix="auto_quant_model", dummy_input=torch.randn(1, 3, 224, 224).to('cpu'), )

2025-02-28 14:56:31,229 - Utils - INFO - successfully created onnx model with 286/1981 node names updated
2025-02-28 14:56:31,726 - Quant - INFO - Layers excluded from quantization: []
2025-02-28 14:56:31,730 - Quant - WARNING - Quantsim export will stop exporting encodings for saving and loading in a future AIMET release.
To export encodings for saving and loading, use QuantizationSimModel's save_encodings_to_json() utility instead.


Quantization Aware Training

In [29]:
print(prepared_model)

GraphModule(
  (patch_embed): Module(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Module(
    (0): Module(
      (blocks): Module(
        (0): Module(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): Module(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (softmax): Softmax(dim=-1)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (module_floordiv_2): FloorDivide()
            (module_mul): Multiply()
            (module_matmul): MatMul()
            (module_add): Add()
            (module_matmul_1): MatMul()
          )
          (drop_path): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
     

Evaluating Accuracy

In [30]:
model

GraphModule(
  (patch_embed): Module(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Module(
    (0): Module(
      (blocks): Module(
        (0): Module(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): Module(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (softmax): Softmax(dim=-1)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (drop_path): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (mlp): Module(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU(approximate='none')
            (drop): Dropout(p=0.0, inplace=False)
         

In [31]:
accuracy = eval_callback(prepared_model)
print(accuracy)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  7.81it/s]

0.86


In [32]:
accuracy = eval_callback(sim.model)
print(accuracy)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:00<00:00,  4.40it/s]

0.85


Fold Batch Normalization layers

In [33]:
print("Layers before BN folding:")
for name, module in prepared_model.named_modules():
    print(name, "->", module)

Layers before BN folding:
 -> GraphModule(
  (patch_embed): Module(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Module(
    (0): Module(
      (blocks): Module(
        (0): Module(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): Module(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (softmax): Softmax(dim=-1)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (module_floordiv_2): FloorDivide()
            (module_mul): Multiply()
            (module_matmul): MatMul()
            (module_add): Add()
            (module_matmul_1): MatMul()
          )
          (drop_path): Identity()
          (norm2): LayerNorm((96,), eps=1e-05, 

In [ ]:
from aimet_torch.batch_norm_fold import fold_all_batch_norms

fold_all_batch_norms(prepared_model, dummy_input.shape, dummy_input=dummy_input)

print("*** After batch norm folding ***")

for name, module in  prepared_model.named_modules():
    print(name, "->", module)




2025-02-28 14:24:24,961 - ConnectedGraph - WARNING - Unable to isolate model outputs.
*** After batch norm folding ***
 -> GraphModule(
  (patch_embed): Module(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): Module(
    (0): Module(
      (blocks): Module(
        (0): Module(
          (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
          (attn): Module(
            (qkv): Linear(in_features=96, out_features=288, bias=True)
            (softmax): Softmax(dim=-1)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=96, out_features=96, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (module_floordiv_2): FloorDivide()
            (module_mul): Multiply()
            (module_matmul): MatMul()
            (module_add): Add()
            (module_matmul_1): MatMul()


In [37]:
from aimet_common.defs import QuantScheme
from aimet_torch.quantsim import QuantizationSimModel

dummy_input = torch.rand(1, 3, 224, 224).to(device)    # Shape for each ImageNet sample is (3 channels) x (224 height) x (224 width)


sim = QuantizationSimModel(model= prepared_model,
                           quant_scheme=QuantScheme.post_training_tf_enhanced,
                           dummy_input=dummy_input,
                           default_output_bw=8,
                           default_param_bw=8)

2025-02-28 14:59:49,202 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-02-28 14:59:49,347 - Quant - INFO - No config file provided, defaulting to config file at /home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/aimet_common/quantsim_config/default_config.json
2025-02-28 14:59:49,370 - Quant - INFO - Unsupported op type Squeeze
2025-02-28 14:59:49,370 - Quant - INFO - Unsupported op type Mean
2025-02-28 14:59:49,401 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:default
2025-02-28 14:59:49,406 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-02-28 14:59:49,407 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-02-28 14:59:49,407 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
202

In [38]:
print(sim.model)

GraphModule(
  (patch_embed): Module(
    (proj): QuantizedConv2d(
      3, 96, kernel_size=(4, 4), stride=(4, 4)
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
    (norm): QuantizedLayerNorm(
      (96,), eps=1e-05, elementwise_affine=True
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): None
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
  )
  (pos_drop): QuantizedDropout(
    p=0.0, inplace=Fa

In [41]:
print(sim.model)

GraphModule(
  (patch_embed): Module(
    (proj): QuantizedConv2d(
      3, 96, kernel_size=(4, 4), stride=(4, 4)
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
    (norm): QuantizedLayerNorm(
      (96,), eps=1e-05, elementwise_affine=True
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): None
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
  )
  (pos_drop): QuantizedDropout(
    p=0.0, inplace=Fa

In [42]:
print(sim)

-------------------------
Quantized Model Report
-------------------------
GraphModule(
  (patch_embed): Module(
    (proj): QuantizedConv2d(
      3, 96, kernel_size=(4, 4), stride=(4, 4)
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=False)
      )
    )
    (norm): QuantizedLayerNorm(
      (96,), eps=1e-05, elementwise_affine=True
      (param_quantizers): ModuleDict(
        (weight): QuantizeDequantize(shape=(), qmin=-128, qmax=127, symmetric=True)
        (bias): None
      )
      (input_quantizers): ModuleList(
        (0): None
      )
      (output_quantizers): ModuleList(
        (0): QuantizeDequantize(shape=(), qmin=0, qmax=255, symmetric=Fal

In [58]:
def pass_calibration_data(model, dataloader, device):
    batch_size = dataloader.batch_size  # Ensure batch_size is correct

    model.eval()
    samples = 1000  # Limit samples for calibration

    batch_cntr = 0
    with torch.no_grad():
        for input_data in dataloader:  # No need for (input_data, target_data)
            inputs_batch = input_data.to(device)  # Move input to device
            model(inputs_batch)  # Forward pass

            batch_cntr += 1
            if (batch_cntr * batch_size) > samples:
                break


In [55]:
dataloader.batch_size

32

In [59]:
for batch in dataloader:
    print(type(batch), len(batch))  # Check if batch has more than 2 elements
    break


<class 'torch.Tensor'> 32


In [60]:
sim.compute_encodings(forward_pass_callback=lambda model: pass_calibration_data(model, dataloader, device))


In [62]:
accuracy = eval_callback(sim.model)
print(accuracy)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:01<00:00,  3.15it/s]

0.85


Apply CLE and BC

In [74]:
import copy

copied_model = copy.deepcopy(prepared_model)


In [75]:
from aimet_torch.cross_layer_equalization import equalize_model

equalize_model(copied_model, input_shapes=(1, 3, 224, 224))

2025-02-28 15:55:15,386 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-02-28 15:55:26,445 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-02-28 15:55:26,519 - CrossLayerEqualization - INFO - High Bias folding is not supported for models without BatchNorm Layers


In [76]:
from aimet_common.defs import QuantScheme
from aimet_torch.quantsim import QuantizationSimModel

dummy_input = torch.rand(1, 3, 224, 224).to(device)    # Shape for each ImageNet sample is (3 channels) x (224 height) x (224 width)


cross_sim = QuantizationSimModel(model= copied_model,
                           quant_scheme=QuantScheme.post_training_tf_enhanced,
                           dummy_input=dummy_input,
                           default_output_bw=8,
                           default_param_bw=8)

cross_sim.compute_encodings(forward_pass_callback=lambda model: pass_calibration_data(model, dataloader, device))


2025-02-28 15:55:48,041 - ConnectedGraph - WARNING - Unable to isolate model outputs.
2025-02-28 15:55:48,166 - Quant - INFO - No config file provided, defaulting to config file at /home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/aimet_common/quantsim_config/default_config.json
2025-02-28 15:55:48,187 - Quant - INFO - Unsupported op type Squeeze
2025-02-28 15:55:48,187 - Quant - INFO - Unsupported op type Mean
2025-02-28 15:55:48,210 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:default
2025-02-28 15:55:48,214 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-02-28 15:55:48,214 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
2025-02-28 15:55:48,215 - Quant - WARNING - Multiple input ops exist, traversal to find closest producer is performed based on the first input
202

In [77]:
accuracy = eval_callback(cross_sim.model)
print(accuracy)

Dataset size 100 DataLoader 4


100%|██████████| 4/4 [00:04<00:00,  1.24s/it]

0.85


In [79]:
from aimet_torch.v1.quantsim import QuantParams
from aimet_torch.bias_correction import correct_bias


bc_params = QuantParams(weight_bw=8, act_bw=8, round_mode="nearest",
                        quant_scheme=QuantScheme.post_training_tf_enhanced)

correct_bias(copied_model, bc_params, num_quant_samples=16,
             data_loader=dataloader, num_bias_correct_samples=16)

Traceback (most recent call last):
  File "/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/fx/graph_module.py", line 274, in __call__
    return super(self.cls, obj).__call__(*args, **kwargs)  # type: ignore[misc]
  File "/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/bmw/anaconda3/envs/swin_bhanu/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1527, in _call_impl
    return forward_call(*args, **kwargs)
  File "<eval_with_key>.29", line 9, in forward
    getitem_3 = getattr_1[3];  getattr_1 = None
IndexError: tuple index out of range

Call using an FX-traced Module, line 9 of the traced Module's generated forward function:
    getitem_2 = getattr_1[2]
    getitem_3 = getattr_1[3];  getattr_1 = None

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE
    patch_embed_proj = self.patch_embed.proj(x);  x = Non

IndexError: tuple index out of range

In [ ]:
import logging

from tqdm import tqdm
import torch
from torch import nn, optim

def _train_loop( model: nn.Module, criterion: torch.nn.modules.loss, optimizer: torch.optim,
                    max_iterations: int, current_epoch: int, max_epochs: int,
                    debug_steps: int = 1000, use_cuda: bool = False):
        """
        Train the specified model using the ImageNet dataset for one epoch.
        :param model: The model to train.
        :param criterion: loss function to optimize
        :param optimizer: optimizer function
        :param max_iterations: total number of iterations to train
        :param current_epoch: current epoch for model training, used for reporting logs in debug steps
        :param max_epochs: total epoch for model training, used for reporting logs in debug steps
        :param debug_steps: number of training iterations to report accuracy/loss metrics
        :param use_cuda: If True then use GPU device

        :return: None
        """
        # pylint: disable-msg=too-many-locals
        device = torch.device('cpu')
        if use_cuda:
            if torch.cuda.is_available():
                device = torch.device('cuda')
            else:
                logger.error('use_cuda is selected but no cuda device found.')
                raise RuntimeError("Found no CUDA Device while use_cuda is selected")

        # switch to training mode
        model = model.to(device)
        model.train()

        avg_loss = 0.0

        for i, (images, target) in tqdm(enumerate(dataloader), total=max_iterations):
            if i == max_iterations:
                break

            images = images.to(device)
            target = target.to(device)

            # compute model output
            output = model(images)
            loss = criterion(output, target)
            avg_loss += loss.item()

            # compute gradient and do SGD step
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (i+1) % debug_steps == 0:
                eval_accuracy = self._evaluator.evaluate(model, use_cuda=use_cuda)
                logger.info('Epoch #%d/%d: iteration #%d/%d: Global Avg Loss=%f, Eval Accuracy=%f',
                            current_epoch, max_epochs, i, max_iterations,
                            avg_loss / i, eval_accuracy)
                # switch to training mode after evaluation
                model.train()

        eval_accuracy = self._evaluator.evaluate(model, use_cuda=use_cuda)
        print("eval : ", eval_accuracy)
        logger.info('At the end of Epoch #%d/%d: Global Avg Loss=%f, Eval Accuracy=%f',
                    current_epoch, max_epochs, avg_loss / max_iterations, eval_accuracy)

def train(model: nn.Module, max_epochs: int = 20, learning_rate: int = 0.1,
              weight_decay: float = 1e-4, decay_rate: float = 0.1, learning_rate_schedule: list = None,
              debug_steps: int = 1000, use_cuda: bool = False):
        """
        Train the specified model using the ImageNet dataset.
        :param model: The model to train.
        :param max_epochs: Maximum number of epochs to use in training.
        :param learning_rate: Learning rate
        :param weight_decay: Weight decay
        :param decay_rate: Multiplicative factor of learning rate decay
        :param learning_rate_schedule: Adjust the learning rate based on the number of epochs
        :param debug_steps: number of training iterations to report accuracy/loss metrics
        :param use_cuda: If True then use GPU device

        :return: None
        """
        max_iterations = len(self._train_loader)

        loss = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)

        if learning_rate_schedule:
            learning_rate_scheduler = optim.lr_scheduler.MultiStepLR(optimizer, learning_rate_schedule,
                                                                     gamma=decay_rate)

        for current_epoch in range(max_epochs):
            self._train_loop(model, loss, optimizer, max_iterations, current_epoch + 1, max_epochs, debug_steps,
                             use_cuda)

            if learning_rate_schedule:
                learning_rate_scheduler.step()

In [69]:
train( model =sim.model , max_epochs=  1 , learning_rate= 5e-7 , learning_rate_schedule= [5,10], use_cuda= True)

NameError: name 'self' is not defined